# 03 · Could we really have known that on Monday?

The most useful check is a simple one. Take the same stocking decision and ask which information actually existed when it was made. An origin is the last completed sales day. If Monday is the origin, Tuesday is day 1 and Wednesday is day 2. Monday sales are counted in lag 0. Tuesday sales are an outcome we learn later.

This notebook builds a few training rows by hand before trusting the full feature builder. We will purposely try an invalid date alignment and look at what it accidentally reveals.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((path for path in (cwd, *cwd.parents) if (path / "configs/config.yaml").is_file()), None)
if root is None:
    raise RuntimeError("Open the notebook from the project root or from notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_audit import load_config
from src.features import KEY, OriginStore, model_features, phase_origins

config_path = root / "configs/config.yaml"
config = load_config(config_path)
processed = root / config["data"]["processed_dir"]
focus = config["data"]["stores"][0]
if not (processed / "panel" / f"{focus}.parquet").is_file():
    raise FileNotFoundError("Run Notebook 01 before building forecast features.")

store = OriginStore.from_processed(processed, focus, config)
phase = config["splits"]["training"]
first = pd.Timestamp(phase["start"])
days = pd.date_range(first + pd.Timedelta(days=40), periods=35)
base = store.origins(days)
first_item = store.items[0]
display(base.loc[base["item_id"].eq(first_item), [*KEY, "lag_0", "lag_1", "lag_7",
    "mean_28", "zero_share_28", "days_since_sale"]].head(5).round({"mean_28": 2, "zero_share_28": 2}))

## 1. Shift the right way

A rolling window describes the last fixed number of completed days. A missing lag means the recorded history does not go far enough back. A lag of zero units is a recorded observation. To test the off-by-one risk, I compare a small feature row with the raw sales dates around it. I also try shifting the series toward the future to see what an incorrect join or shift would reveal.

In [ ]:
origin = days[0]
observed = pd.Series(store.sales[0], index=store.days, name="recorded_units")
row = base.loc[base["item_id"].eq(first_item) & base["origin"].eq(origin)].iloc[0]
trace = observed.loc[origin - pd.Timedelta(days=7):origin + pd.Timedelta(days=2)].to_frame()
trace["known_by_origin"] = trace.index <= origin
display(trace)
comparison = pd.DataFrame([
    {"quantity": "lag 0", "feature value": row["lag_0"], "manual value": observed.loc[origin]},
    {"quantity": "lag 7", "feature value": row["lag_7"], "manual value": observed.loc[origin - pd.Timedelta(days=7)]},
    {"quantity": "28-day mean", "feature value": row["mean_28"],
     "manual value": observed.loc[origin - pd.Timedelta(days=27):origin].mean()},
])
np.testing.assert_allclose(comparison["feature value"], comparison["manual value"])
display(comparison.round(3))
print(f"Using shift(-1) as a feature here gives {observed.shift(-1).loc[origin]:g}. "
      f"It equals the next day's observed sales, {observed.loc[origin + pd.Timedelta(days=1)]:g}.")

## 2. Which price date matters?

Notebook 01 showed that a price attached to a target date can belong to a week still in progress at the origin. The weekly CSV has no publication timestamp. I start with the completed-week rule, test the price feature against a price-free model on development, and leave missing past prices null. A retailer with timestamped planned prices could change the availability rule.

A good price feature needs two kinds of evidence: the value and the date that makes it eligible. Here I inspect the age of eligible prices and compare with the unsafe idea of using the price for the week containing the origin.

In [ ]:
show = [*KEY, "asof_price", "price_source_week", "price_week_end", "relative_price_change", "price_age_days"]
priced = base.loc[base["item_id"].eq(first_item), show]
display(priced.head(10).round({"asof_price": 3, "relative_price_change": 3}))
assert base["price_week_end"].dropna().le(base.loc[base["price_week_end"].notna(), "origin"]).all()
display(base["price_age_days"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame("days since eligible price week"))
print(f"{base['price_missing'].mean():.2%} of the inspected rows have no eligible listed price.")

## 3. One target day and one protection period answer different questions

A day-8 forecast predicts the units on the eighth day only. An eight-day inventory target covers demand from day 1 through day 8. Repeating last week's pattern is a usable baseline for either task, but the cumulative version adds its eight daily point forecasts. With uncertain demand, adding the daily 90th percentiles does not generally give the 90th percentile of the sum.

Window eligibility also matters. The last few origins can have a two-day label and lack a complete eight-day label. Count complete windows before comparing horizons. Compared scores will use common origins. Replay still gets recommendations for the final days, with unavailable outcomes left blank.

In [ ]:
sample = base.loc[base["origin"].eq(origin) & base["item_id"].eq(first_item)]
rows = []
for family, horizon in [("daily", 2), ("daily", 8), ("cumulative", 2), ("cumulative", 8)]:
    design = store.design(sample, horizon, family=family, labels=phase)
    rows.append(design.assign(family=family)[["family", "origin", "target_date", "horizon", "actual",
        "seasonal_naive", "target_weekday", "target_snap", "window_snap_days"]]
        if family == "cumulative"
        else design.assign(family=family)[["family", "origin", "target_date", "horizon", "actual",
        "seasonal_naive", "target_weekday", "target_snap"]])
display(pd.concat(rows, ignore_index=True))
display(pd.DataFrame([
    {"phase": name, **{f"complete H={h}": len(phase_origins(block, h))
                      for h in config["forecast"]["cumulative_horizons"]}}
    for name, block in config["splits"].items()
]))

## 4. Try to break the data boundary

If changing future sales or prices changes an earlier feature, a good-looking model score would be unreliable. Use a small artificial series so a reader can see the changed rows. Separately, request origins with dates missing between them. Lag 7 should still mean seven calendar days, even if six origins were skipped. Scheduled calendar fields are allowed to describe the target date, so we leave them fixed during the perturbation.

In [ ]:
example_days = pd.date_range("2024-01-01", periods=64)
panel = pd.DataFrame({
    "store_id": "CA_1", "item_id": "Example", "dept_id": "FOODS_1",
    "date": example_days, "sales": np.arange(len(example_days)) % 6,
})
calendar = pd.DataFrame({
    "date": example_days, "weekday": example_days.weekday, "month": example_days.month,
    "event_name_1": None, "event_name_2": None,
    "snap_CA": False, "snap_TX": False, "snap_WI": False,
})
prices = pd.DataFrame({
    "store_id": "CA_1", "item_id": "Example", "wm_yr_wk": [1, 2, 3],
    "week_end": example_days[[29, 36, 43]], "sell_price": [2.0, 3.0, 4.0],
})
seek = example_days[[33, 35]]
original = OriginStore(panel, calendar, prices, config)
before = original.origins(seek)
dense = original.origins(example_days[33:36])
pd.testing.assert_frame_equal(before, dense.loc[dense["origin"].isin(seek)].reset_index(drop=True))

panel.loc[panel["date"].gt(seek[-1]), "sales"] = 50_000
prices.loc[prices["week_end"].gt(seek[-1]), "sell_price"] = 5_000
changed = OriginStore(panel, calendar, prices, config)
after = changed.origins(seek)
pd.testing.assert_frame_equal(before, after)
for family in ("daily", "cumulative"):
    pd.testing.assert_frame_equal(
        original.design(before, 8, family=family), changed.design(after, 8, family=family)
    )
assert "actual" not in model_features(original.design(before, 8, family="daily"), use_price=True)
display(before[["origin", "lag_0", "lag_7", "asof_price", "price_week_end"]])
print("PASS: later sales and prices did not alter earlier features, and sparse origins kept calendar lags.")

## 5. What I would carry forward

- Use recent lags and rolling sales for level and variation. Store an explicit zero share and time since last sale because two items with the same mean can have different sales frequencies.
- Keep the scheduled target-day and protection-window calendar fields. Their historical availability is an assumption about the calendar schedule. The pricing proxy is separately checked.
- Use direct daily and cumulative targets, keep their scores separate, and compare horizons on the same observed origins.
- Retain the origin key, target date, and price week in the feature view for inspection. The estimator receives an explicit list of features, so keys, outcomes and provenance dates cannot slip in as predictors.

The latest label after the source's observed sales end is unknown. Notebook 04 will keep that distinction in the exported replay forecasts.

To materialize the same feature interface, use the command under **Reproduce Part 2** in the [README](../README.md). The export is partitioned by store, family and horizon. It includes price provenance and leaves the outcomes out.